<a href="https://colab.research.google.com/github/nyp-sit/iti121-2026s2/blob/main/L3/04b_asl_predictions_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Applied Deep Learning

# 4b. Deploying Your Model
In this exercise, we will load our saved model and use it to predict the correct letters of the sign language alphabet.

## 4b.1 Objectives

* Load an already-trained model from disk
* Reformat images for a model trained on images of a different format
* Perform inference with new images never seen during training
* Convert the model output into a predicted ASL letter

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.io as tv_io
import torchvision.transforms.v2 as transforms
import torchvision.transforms.functional as F
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 4b.2 Loading the Model
Now that we're in a new notebook, let's load the saved model that we trained.

Since our model uses a custom module, we need the same `MyConvBlock` definition that was used when the model was trained.

We define the class directly in the notebook. Alternatively, we can also load it from an external `utils.py` file.

In [ ]:
class MyConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout_p):
        kernel_size = 3
        super().__init__()

        self.model = nn.Sequential(
            nn.Conv2d(
                in_ch,
                out_ch,
                kernel_size,
                stride=1,
                padding=1
            ),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.MaxPool2d(2, stride=2)
        )

    def forward(self, x):
        return self.model(x)

Now that we have a definition for `MyConvBlock`, we can recreate the same model architecture and use `torch.load` to load the trained parameters from disk.

The model filename has `.pth` extension, eg. `model.pth`.

We store the model's `state_dict` and a small amount of metadata rather than serialising the complete Python model object.

In [ ]:
import os
os.getcwd()

In [ ]:
IMG_HEIGHT = 28
IMG_WIDTH = 28
IMG_CHS = 1
N_CLASSES = 24

flattened_img_size = 75 * 3 * 3

model = nn.Sequential(
    MyConvBlock(IMG_CHS, 25, 0),
    MyConvBlock(25, 50, 0.2),
    MyConvBlock(50, 75, 0),
    nn.Flatten(),
    nn.Linear(flattened_img_size, 512),
    nn.Dropout(0.3),
    nn.ReLU(),
    nn.Linear(512, N_CLASSES)
)

checkpoint_path = "asl_cnn_augmented_checkpoint.pth"

if checkpoint_path is None:
    raise FileNotFoundError(
        "Could not find model.pth or asl_cnn_augmented_checkpoint.pt. "
        "Run the 04a augmentation practical first and save the trained model."
    )

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

if not (isinstance(checkpoint, dict) and "model_state_dict" in checkpoint):
    raise ValueError(
        "This notebook expects the independent Week 2 checkpoint format "
        "containing 'model_state_dict'."
    )

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

print("Loaded:", checkpoint_path)
model

We can also verify that the model parameters are on the expected device.

In [ ]:
next(model.parameters()).device

## 4b.3 Preparing an Image for the Model

It's now time to use the model to make predictions on new images that it's never seen before. This is also called **inference**.

We will take some example PNG files from the public **Sign Language MNIST test set**.

The model was trained on 28 × 28 grayscale images. Whenever we make predictions with a trained model, the input must be converted to the same basic format used during training.

### 4b.3.1 Showing the Images

When we use our model to make predictions on new images, it will be useful to show the image as well. We can use Matplotlib to do this.

In [ ]:
# If kagglehub is not installed in a new environment, install it once with:
# %pip install -q kagglehub

import kagglehub

dataset_path = Path(
    kagglehub.dataset_download("datamunge/sign-language-mnist")
)

TEST_CSV = dataset_path / "sign_mnist_test" / "sign_mnist_test.csv"
if not TEST_CSV.exists():
    TEST_CSV = dataset_path / "sign_mnist_test.csv"

test_df = pd.read_csv(TEST_CSV)

asl_image_dir = Path.cwd() / "data" / "asl_images"
asl_image_dir.mkdir(parents=True, exist_ok=True)

def save_example_letter(raw_label, filename):
    row = test_df[test_df["label"] == raw_label].iloc[0]
    pixels = row.drop("label").to_numpy(dtype=np.uint8).reshape(28, 28)

    image_tensor = torch.tensor(pixels).unsqueeze(0)
    tv_io.write_png(image_tensor, str(asl_image_dir / filename))

# A = raw label 0, B = raw label 1
save_example_letter(0, "a.png")
save_example_letter(1, "b.png")

def show_image(image_path):
    image = tv_io.read_image(
        str(image_path),
        tv_io.ImageReadMode.GRAY
    )
    plt.imshow(image.squeeze(0), cmap="gray")
    plt.axis("off")
    plt.show()

print("Example images created in:", asl_image_dir)

In [ ]:
show_image(asl_image_dir / "b.png")

### 4b.3.2 Scaling the Images

The images in our training dataset were 28 × 28 pixels and grayscale. We need to make sure that images used for prediction are converted to the same size and grayscale format.

TorchVision provides `read_image` and `ImageReadMode` for loading image files consistently.

In [ ]:
image = tv_io.read_image(
    str(asl_image_dir / "b.png"),
    tv_io.ImageReadMode.GRAY
)
image

Let's look at the shape of the image.

In [ ]:
image.shape

We can use TorchVision transforms to put the data into the form our model expects.

We will:

* convert the image to floating-point values with `ToDtype`;
  * `scale=True` converts values from `[0, 255]` to approximately `[0, 1]`;
* resize the image to `28 × 28`;
* ensure that the image is grayscale.

The sample images created above are already grayscale and 28 × 28, but keeping these preprocessing steps makes the prediction pipeline suitable for other image files as well.

In [ ]:
IMG_WIDTH = 28
IMG_HEIGHT = 28

preprocess_trans = transforms.Compose([
    transforms.ToDtype(torch.float32, scale=True),
    transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
    transforms.Grayscale()
])

Let's test `preprocess_trans` on an image to make sure it works correctly:

In [ ]:
processed_image = preprocess_trans(image)
processed_image

The numbers look correct, but how about the shape?

In [ ]:
processed_image.shape

Next, let's plot the image to see if it looks like what we trained on.

In [ ]:
plot_image = F.to_pil_image(processed_image)
plt.imshow(plot_image, cmap='gray')

### 4b.4 Making Predictions

Okay, now we're ready to predict. Our model still expects a **batch** of images.

`unsqueeze(0)` adds a dimension of size 1 at the first position. Because the first dimension is normally the batch dimension, this changes:

```text
[channel, height, width]
```

into:

```text
[batch, channel, height, width]
```

In [ ]:
batched_image = processed_image.unsqueeze(0)
batched_image.shape

Next, we should make sure the input tensor is on the same `device` as the model.

In [ ]:
batched_image_gpu = batched_image.to(device)
batched_image_gpu.device

Now we're ready to pass it to the model. During inference, we use `torch.no_grad()` because we do not need to calculate gradients.

In [ ]:
model.eval()

with torch.no_grad():
    output = model(batched_image_gpu)

output

### 4b.4.1 Understanding the Prediction

The prediction is a 24-element array of **logits**: one value for each possible class.

The larger the logit, the stronger the model's score for that class. We can use `argmax` to find the index of the largest value.

In [ ]:
prediction = output.argmax(dim=1).item()
prediction

Each element of the output array represents one possible letter in the sign language alphabet. Remember that **J** and **Z** are not included because they require movement, while this dataset contains still images.

In [ ]:
# Alphabet does not contain J or Z because they require movement
alphabet = "abcdefghiklmnopqrstuvwxy"

We can now pass the predicted class index to find the corresponding letter.

In [ ]:
alphabet[prediction]

#### Exercise

Let's put everything in a function so that we can make predictions directly from an image file. Implement it below using the same steps we have just followed. If you need help, reveal the solution afterwards.

In [ ]:
def predict_letter(file_path):
    # Show image
    # TODO

    # Load and grayscale image
    image = None  # TODO

    # Transform image
    image = None  # TODO

    # Batch image
    image = None  # TODO

    # Send image to correct device
    image = None  # TODO

    # Make prediction
    output = None  # TODO

    # Find max index
    prediction = None  # TODO

    # Convert prediction to letter
    predicted_letter = None  # TODO

    return predicted_letter

#### Solution

Run the solution cell after attempting the exercise.

In [ ]:
# @title
# SOLUTION
def predict_letter(file_path):
    show_image(file_path)

    image = tv_io.read_image(
        str(file_path),
        tv_io.ImageReadMode.GRAY
    )

    image = preprocess_trans(image)
    image = image.unsqueeze(0)
    image = image.to(device)

    model.eval()
    with torch.no_grad():
        output = model(image)

    prediction = output.argmax(dim=1).item()

    # Convert prediction to letter
    predicted_letter = alphabet[prediction]

    return predicted_letter

In [ ]:
predict_letter(asl_image_dir / "b.png")

Let's also use the function with the **A** image in the `asl_images` folder:

In [ ]:
predict_letter(asl_image_dir / "a.png")

## 4b.5 Summary

You have used the saved model to make predictions on new image files. You can take your own ASL letter images, place them in the `data/asl_images` folder, and test the prediction function.

Keep in mind that real camera images can differ from the training data in lighting, background, hand position, scale, and image quality. These differences can affect prediction accuracy.

---
### Dataset attribution

**Sign Language MNIST** — Kaggle dataset by *datamunge*  
License: **CC0: Public Domain**

